# Bölüm 11 · Korelasyon Analizi

[Atölye](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b11/) · [Veri ve lisans](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b11/VERI.md) · [Görevler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b11/GOREVLER.md) · [Raporlama](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b11/RAPORLAMA.md) · [Çözümler](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b11/COZUMLER.md)

**Başlangıç:** Google hesabınızla açın, **Dosya → Drive'a bir kopyasını kaydet** seçin. CPU yeterlidir. Hücreleri sırayla veya **Çalışma zamanı → Tümünü çalıştır** ile yürütün. İlk kurulum internet ve birkaç dakika gerektirebilir. Özellikle bootstrap ve yapısal modeller daha uzun sürebilir.

Bu defter, aynı GitHub deposundaki ana Python analizini ve bölümün ileri eklerini çalıştırır. Dosya yüklemeniz veya önce başka bölümü elle yürütmeniz gerekmez. Gerçek veri, öğretim alt kümesi ve simülasyon ayrımı veri rehberinde açıklanır; bunları aynı örneklem türü saymayın. R/SPSS/Amos burada çalıştırılmaz.

## Üç çekirdek kazanım ve ileri rota

1. Birlikte değişimi saçılım grafiğiyle inceler.
2. Pearson ve Spearman seçimini hedefle gerekçelendirir.
3. Korelasyon, uyum ve nedenselliği ayırır.

**İleri rota:** Fisher aralığı, eşli bootstrap ve etki duyarlılığı; Bölüm 4 belirsizlik temellerinden sonra.

**Ç1 · Çıkış sorusu:** İkinci cihaz her ölçümde birinciden 10 birim fazla veriyorsa r=1 olması tam uyum demek midir?

Yanıtınızı çalışma sonunda öğrenci karar günlüğüne yazın. [Rubrik ve teslim rehberi](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b11/RUBRIK.md).

## 1 · Kaynakları hazırla

Sabit bir GitHub sürümündeki öğrenci paketi SHA256 ile doğrulanır. Her çalıştırma ayrı bir geçici çalışma kopyası açar; önceki sonuçlarınız silinmez. Kod ve veri, bu doğrulanan sürümden gelir. Oturum kapanmadan sonuçları indirin.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, urllib.request, zipfile, io, tempfile, subprocess, importlib.metadata
DATA_COMMIT = '23acf4be23823aa139523c8f4db34cff248c285d'
ARCHIVE_SHA256 = 'ce6440f5ee3b4dde9709e1fb250e7b7d8ac86be101b6ababacdab240443560f6'
ARCHIVE_URL = f"https://raw.githubusercontent.com/ibrahimguney/ileri-istatistik-uygulamalari/{DATA_COMMIT}/surumler/22-bolum/ogrenci-paketi-22-bolum.zip"
cache = Path.cwd()/"kitap-colab-kaynak.zip"
if cache.exists():
    raw = cache.read_bytes()
else:
    try:
        with urllib.request.urlopen(ARCHIVE_URL, timeout=60) as response: raw=response.read()
    except Exception as exc:
        raise RuntimeError("Kaynak indirilemedi. İnterneti kontrol edip hücreyi yeniden çalıştırın: "+ARCHIVE_URL) from exc
if hashlib.sha256(raw).hexdigest()!=ARCHIVE_SHA256:
    raise ValueError("Öğrenci paketi SHA256 denetimi başarısız. Kontrolü kaldırmayın; yerel kitap-colab-kaynak.zip dosyasını doğrulayın.")
cache.write_bytes(raw)
work = Path(tempfile.mkdtemp(prefix="b11-atolye-",dir=Path.cwd()))
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert len(z.infolist())<10000 and sum(i.file_size for i in z.infolist())<134217728
    for member in z.infolist():
        target=(work/member.filename).resolve()
        if work.resolve() not in target.parents or ((member.external_attr>>16)&0o170000)==0o120000:
            raise ValueError("Güvenli olmayan arşiv yolu")
    z.extractall(work)
root = work/"ileri-istatistik-22-bolum"
assert (root/"BOLUM_ESLESMESI.json").is_file()
application=root/'uygulamalar/b11-korelasyon'
modules=['uygulamalar/b11-korelasyon/esli-bootstrap']
expected_main=json.loads((root/"bolumler/b11/beklenen.json").read_text())
expected_extra={m:json.loads((root/m/"sonuclar/ozet.json").read_text()) for m in modules}
print("Kaynak doğrulandı; Bölüm 11 çalışma kopyası hazır.")

## 2 · Analiz ortamını hazırla

Sınanmış paket sürümleri kullanılır. Mevcut çekirdek bu sürümlerdeyse doğrudan kullanılır. Farklıysa analizler ayrı bir Python ortamında kurulur; Colab çekirdeğindeki paketler topluca değiştirilmez. Kurulum başarısız olursa hücredeki hatayı inceleyin. Python 3.12 veya üzeri çalışma zamanı önerilir. GPU gerekmez.

In [ ]:
FORCE_ISOLATED=globals().get("FORCE_ISOLATED",False)
PINS={'numpy': '2.5.4', 'scipy': '1.18.1', 'pandas': '3.1.0', 'matplotlib': '3.11.2', 'statsmodels': '0.15.0', 'PyMuPDF': '1.26.6'}
def package_versions(executable):
    check="import importlib.metadata,json; names="+repr(list(PINS))+";print(json.dumps({n:importlib.metadata.version(n) for n in names}))"
    attempt=subprocess.run([str(executable),"-c",check],capture_output=True,text=True)
    return json.loads(attempt.stdout) if attempt.returncode==0 else {}
python=Path(sys.executable)
if FORCE_ISOLATED or package_versions(python)!=PINS:
    environment=Path.cwd()/".kitap-colab-python"
    if not (environment/"bin/python").exists():
        subprocess.run([sys.executable,"-m","venv",str(environment)],check=True)
    python=environment/"bin/python"
    if package_versions(python)!=PINS:
        subprocess.run([str(python),"-m","pip","install",*[name+"=="+version for name,version in PINS.items()]],check=True)
assert package_versions(python)==PINS,"Paket sürümleri sınanmış ortamla eşleşmiyor"
versions=package_versions(python)
run_env={**os.environ,"MPLBACKEND":"Agg","MPLCONFIGDIR":str(work/"mpl"),"OPENBLAS_NUM_THREADS":"1","OMP_NUM_THREADS":"1","PYTHONDONTWRITEBYTECODE":"1"}
(work/"logs").mkdir(exist_ok=True)
def run_script(relative):
    print("Çalışıyor:",relative,flush=True)
    result=subprocess.run([str(python),str(root/relative)],cwd=root,env=run_env,capture_output=True,text=True,timeout=900)
    (work/"logs"/(relative.replace("/","__")+".log")).write_text(result.stdout+"\n"+result.stderr)
    if result.returncode: raise RuntimeError(result.stdout[-4000:]+"\n"+result.stderr[-4000:])
    print("Tamamlandı.")
print("Analiz ortamı hazır:",versions)

## Ana analiz

Kaynak ve yöntem için yukarıdaki veri rehberini okuyun. Ana analiz ve sonradan eklenen duyarlılık uygulamaları ayrı çalıştırılır. Sonuçlar önceden gösterilen tabloların kopyası olarak değil, kod yeniden yürütülerek üretilir.

In [ ]:
before_figures=set((root/"assets_figures").glob("*.pdf"))
run_script("bolumler/b11/analiz.py")
main_result=json.loads((application/"sonuclar/ozet.json").read_text())
from IPython.display import display, JSON, HTML, Image
print("Ana sonuçlar — ayrıntıları açmak için okları kullanın:")
display(JSON(main_result,expanded=False))

## İleri uygulamalar ve duyarlılık

Her ekin açıklaması atölye sayfasındadır. Bu sonuçları ana analizle veya birbirinden farklı tahmin hedefleriyle karıştırmayın. Hazır kontrol betikleri varsa önce onlar, ardından ek analiz çalışır.

In [ ]:
extra_results={}
for module in modules:
    if (root/module/"dogrula.py").is_file(): run_script(module+"/dogrula.py")
    run_script(module+"/analiz.py")
    extra_results[module]=json.loads((root/module/"sonuclar/ozet.json").read_text())
    print(module)
    display(JSON(extra_results[module],expanded=False))

## Sonuçları referanslarla karşılaştır

Kitabın sayısal referansları, önceki sürümden değişmeden alınır. Sürüm/zaman gibi hesap dışı alanlar mevcut karşılaştırma politikasına göre dışlanır. Hata oluşursa veri, sürüm ve günlük dosyalarını kontrol edin; referansları sonuca uydurmayın. Bu kontrol bilimsel varsayımları veya SPSS yürütmesini doğrulamaz.

In [ ]:
sys.path.insert(0,str(root/"tools"))
from dogrula_22 import compare
checks={"main":compare(expected_main,main_result)}
for module in modules:
    checks[module]=compare(expected_extra[module],json.loads((root/module/"sonuclar/ozet.json").read_text()))
assert all(not differences for differences in checks.values()),checks
print("Ana analiz ve",len(modules),"ileri ek için sayısal referanslar eşleşti.")

## Grafikler ve özet tablolar

Grafikler bu çalıştırmada üretilir. Ön koşul olarak çalıştırılan Bölüm 18'in şekilleri burada tekrar gösterilmez. CSV tablolarının ilk 12 satırı görüntülenir; tam dosyalar çıktı paketindedir. Şekil sayısı bölüme göre değişir.

In [ ]:
import csv,html
new_figures=sorted(set((root/"assets_figures").glob("*.pdf"))-before_figures)
for figure in new_figures:
    png=figure.with_suffix(".png")
    renderer="import fitz,sys;d=fitz.open(sys.argv[1]);d[0].get_pixmap(matrix=fitz.Matrix(1.4,1.4)).save(sys.argv[2])"
    subprocess.run([str(python),"-c",renderer,str(figure),str(png)],check=True)
    print(figure.name)
    display(Image(filename=str(png)))
for table in sorted((application/"sonuclar").glob("*.csv"))[:3]:
    with table.open(encoding="utf-8-sig",newline="") as f: rows=list(csv.reader(f))
    content="<h4>"+html.escape(table.name)+"</h4><table>"
    for i,row in enumerate(rows[:13]):
        tag="th" if i==0 else "td"
        content+="<tr>"+"".join("<"+tag+">"+html.escape(value)+"</"+tag+">" for value in row)+"</tr>"
    display(HTML(content+"</table>"))

## Sıra sizde

# Bölüm 11 — görevler

Güncel Revizyon 14 görevleri; önce kendiniz çözün.

## D1–D3: Hesaplamadan Önce Düşünün

1.  Şekil <a href="#fig:b06-uci" data-reference-type="ref" data-reference="fig:b06-uci">[fig:b06-uci]</a>’nin ham not panelinde ilişkinin yönünü ve diğerlerinden ayrılan kaydı belirleyin. Tek katsayı bu grafikte hangi bilgiyi kaybettirir?

2.  “Puanlar doğrusal biçimde birlikte değişiyor mu?” ve “Yüksek not alanlar diğer dönemde de üst sıralarda mı?” soruları için hangi katsayıyı seçersiniz? İkisini de hesaplamak neden yararlı olabilir?

3.  İlk kayıtta $G1=0$, $G3=11$ bulunması bu kaydı eksik veya hatalı saymak için yeterli midir? Kontrol etmek için hangi kaynak bilgisi gerekir?

Yanıtları önce kendiniz yazın. Bu bölümdeki yeni etkinliklerin gerekçeli çözümleri Ek F’de, s. kitaptaki ilgili sayfa’tedir.

# Rehberli ve Bağımsız Etkinlikler

## G1–G3: Rehberli Uygulama

1.  60 kayıt içinde $G1=12$ ve $G3=12$ için ortalama sıraları hesaplayın. Kaynak sırası 3 olan kaydın sıra çiftini yazın. Yalnızca ilk altı kaydı sıralamak neden aynı sonucu vermez?

2.  Yalnız G1’i $M=5G1$ ile 100’lük ölçeğe dönüştürün. Pearson ve Spearman değişir mi? Sayısal hesap ve kavramsal gerekçeyi birlikte verin.

3.  $s_x=2.5558$, $s_y=1.9923$ kullanarak $b_1=r(s_y/s_x)$ ve $R^2=r^2$ hesaplayın. Bölüm 12 ile karşılaştırın; $r_s^2$ neden aynı açıklama değildir?

## B1–B3: Sonraki 20 Kayıtta Bağımsız Çözüm

Kaynak satırları 61–80’i seçin; ana grubun katsayılarını bu gruba taşımayın.

1.  Grafiği çizin; Pearson ve ortalama sıralı Spearman katsayılarını yeniden hesaplayın. Pearson hesabını $S_{xx}=78.8$, $S_{yy}=85.2$, $S_{xy}=51.4$ ile denetleyin.

2.  Pearson için yaklaşık Fisher %95 aralığını hesaplayın. İlk 60 kaydın aralığıyla karşılaştırıp yalnızca aralıkların örtüşmesine dayanarak bir fark testi yapmanın neden yeterli olmadığını açıklayın.

3.  En fazla 100 kelimelik raporda değişkenleri, seçimi, iki katsayıyı ve iki sınırlılığı belirtin. Bu 20 kaydı “bağımsız dış doğrulama örneklemi” olarak adlandırmayın.

## H1: Hatalı Raporu Düzeltin

Her iddiayı ayrı değerlendirin ve veriyle desteklenen bir alternatif yazın.

## T1–T3: Birlikte Belirsizlik ve Uyum Denetimi

1.  Pearson ve Spearman aralıkları hesaplanırken neden aynı öğrenci satırları birlikte çekilir ve sıralar her çekilişte yeniden hesaplanır? Sütunları ayrı örneklemek ve aynı kişilerin katsayılarını bağımsız Fisher formülüyle karşılaştırmak hangi yapıyı bozar?

2.  $r_s-r$ yaklaşık aralığı $[-.0865,.2018]$ neden yöntem üstünlüğü veya eşdeğerlik kanıtı değildir? İlk satırın olmadığı 7164 bootstrap örneği neden yeni bir doğrulama grubu değildir?

3.  $r(X,X+5)=1$ neden uyum değildir? $Y=X^2$, $X\sim U(-1,1)$ neden sıfır korelasyonun bağımsızlık olmadığını gösterir? Her örnekte hangi hipotez ve hangi grafik gerekir?

## P1: Tekrarlanabilir Duyarlılık Projesi

İlk 60 kayıt için ana analizi, ilk kaydı dışarıda bırakma analizini ve sırayla her bir kaydı dışarıda bırakan 60 karşılaştırmayı üretin. Her alt kümede sıraları yeniden hesaplayın. Pearson ve Spearman’ın en küçük/en büyük değerlerini ve bunlara karşılık gelen kaynak satırlarını bulun. Bir sayfalık raporu veri seçimi, grafik, yöntem, sonuç, sınırlılıklar ve yeniden çalıştırma komutuyla teslim edin.

<div class="center">

| Değerlendirme ölçütü                                             | Puan |
|:-----------------------------------------------------------------|-----:|
| Kaynak atfı, seçim kuralı ve değişmeyen verinin belgelenmesi     |   20 |
| Grafikler, bağlı sıralar ve doğru katsayılar                     |   25 |
| Yeniden sıralanan dışarıda bırakma karşılaştırmaları             |   25 |
| Belirsizlik, silme kararı ve genelleme sınırlarının tartışılması |   20 |
| Kod, sürümler ve yeniden çalıştırma yönergesi                    |   10 |

</div>

# Alıştırmalar

## Kolay Düzey

1.  $r=-1$ neyi gösterir?

2.  Belirlilik katsayısı $r^2$ temel olarak neyi gösterir?

3.  $r=.75$ yön ve güç bakımından nasıl yorumlanır?

4.  $r^2$ hangi aralıkta değer alır?

5.  Pearson korelasyonu hangi tür ilişkiye daha duyarlıdır?

## Orta Düzey

1.  Bir araştırmada çalışma süresi ile sınav puanı arasındaki korelasyon $r=.62$ bulunmuştur. Bu sonucu yön ve güç bakımından açıklayınız.

2.  Bir korelasyon analizinde $r=-.40$ ve $p<.05$ elde edilmiştir. Katsayı ve anlamlılık sonucunu birlikte yorumlayınız.

3.  Pearson korelasyonu sıfıra yakın bulunmuş, ancak saçılım grafiği belirgin eğrisel örüntü göstermiştir. Bu durumu açıklayınız.

4.  Bir araştırmacı iki değişken arasında $r=.50$ bulmuştur. $r^2$ kaçtır ve nasıl yorumlanır?

5.  İki sıralı tutum puanı arasındaki ilişki incelenecektir. Pearson mı Spearman mı daha uygundur? Kısaca açıklayınız.

## İleri Düzey

1.  Bir saçılım grafiğinde tek bir uç gözlemin korelasyonu güçlü biçimde yukarı çektiği görülmüştür. Araştırmacı sonucu raporlamadan önce ne yapmalıdır?

2.  Fiziksel aktivite ile yaşam doyumu arasında $r=.55$ bulunmuştur. Araştırmacının “fiziksel aktivite yaşam doyumunu artırır” demesi neden sorunludur?

3.  Bir korelasyon tablosunda stres ve uyku kalitesi için $r=-.68$, $p=.002$, $n=42$ raporlanmıştır. Kısa akademik yorum yazınız.

4.  Bir veri setinde Pearson korelasyonu $r=.20$, Spearman korelasyonu $r_s=.71$ bulunmuştur. Bu fark neyle açıklanabilir?

5.  İki değişken için $r=.00$ bulunmasına karşın araştırmacı bağımsız olmadıklarından şüphelenmektedir. Sıfır korelasyon hangi durumlarda bağımsızlık anlamına gelmeyebilir?


Önce kendi yorumunuzu yazın; sonra [çözümlerle](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b11/COZUMLER.md) karşılaştırın. Sonuç, belirsizlik, varsayım ve sınırlılıkları [raporlama rehberine](https://ibrahimguney.github.io/ileri-istatistik-uygulamalari/surumler/22-bolum/bolumler/b11/RAPORLAMA.md) göre ayırın.

In [ ]:
yanitlarim={"bulgular":"", "belirsizlik":"", "varsayimlar":"", "sinirliliklar":""}
# Metinleri doldurun; sayısal kontrolü geçmek bu yorumları tamamlamaz.

## Çıktıları indir

Son hücre sonuç JSON/CSV'lerini, grafiklerin PDF/PNG dosyalarını, günlükleri ve kendi yanıtlarınızı paketler. Colab'ın solundaki Dosyalar panelini yenileyin ve aşağıda yazan ZIP'i indirin. Geçici çalışma zamanı kapanmadan bunu yapın; kod Drive'a otomatik erişmez.

## Öğrenci kararı ve kısa savunma

**Karar görevi:** Pearson veya Spearman özetini seçin; aynı satırlar ve grafik altında hedef, katsayı ve belirsizliği karşılaştırın.

Yalnız bir kararı değiştirin. Ana sonucu koruyun; gerekirse yeni bir hesap hücresi ekleyip denemeyi ayrı kaydedin. Sayısal referansları yeni sonuca uydurmayın. Önce/sonra kanıtını sonuç dosyası, tablo veya açık bir hesapla ilişkilendirin. Yapılmamış hesabı yapılmış gibi yazmayın.

**3–5 dakikalık savunma:** Hangi hedefi seçtiniz ve neden? Tam olarak ne değişti? Hangi belirsizlik ve kapsam sınırı kaldı? Karşılaştırma aynı hipotezi mi sınadı?

Aşağıdaki alanları kendi çalışmanızla doldurun. **Tümünü çalıştır**, bu alanları sizin yerinize tamamlamaz. Sayısal kontrollerin geçmesi öğrenci kararının tamamlandığı anlamına gelmez. Alanların dolu olması da kanıtın doğruluğunu otomatik onaylamaz.

In [ ]:
karar_gunlugu = {
    "bolum": 11,
    "gorev": 'Pearson veya Spearman özetini seçin; aynı satırlar ve grafik altında hedef, katsayı ve belirsizliği karşılaştırın.',
    "kararim": "",
    "gerekcem": "",
    "degistirdigim_tek_sey": "",
    "onceki_kanit": "",
    "sonraki_kanit": "",
    "belirsizlik_ve_sinir": "",
    "kisa_savunma": "",
    "cikis_yanitim": ""
}
zorunlu_alanlar = ["kararim","gerekcem","degistirdigim_tek_sey","onceki_kanit","sonraki_kanit","belirsizlik_ve_sinir","kisa_savunma","cikis_yanitim"]
karar_gunlugu["alanlar_dolu"] = all(isinstance(karar_gunlugu[k],str) and bool(karar_gunlugu[k].strip()) for k in zorunlu_alanlar)
karar_gunlugu["durum"] = "Öğretim elemanı incelemesi gerekir" if karar_gunlugu["alanlar_dolu"] else "Öğrenci kararı henüz tamamlanmadı"
print(karar_gunlugu["durum"])
# Bu hücreyi doldurduktan sonra çıktı ZIP'i hücresini yeniden çalıştırın.

In [ ]:
manifest={"chapter":11,"source_commit":DATA_COMMIT,"source_archive_sha256":ARCHIVE_SHA256,"versions":versions,"checks":checks,"answers":yanitlarim,"scope":"Python ana analiz ve ileri ekler; SPSS/R/Amos çalıştırılmadı"}
# OGRETIM_EXPORT_BEGIN
manifest["ogrenci_karari"] = karar_gunlugu
(work/"ogrenci-karari.json").write_text(json.dumps(karar_gunlugu,ensure_ascii=False,indent=2),encoding="utf-8")
# OGRETIM_EXPORT_END
(work/"calisma-kaydi.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
archive=Path.cwd()/"b11-colab-ciktilar.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    z.write(work/"ogrenci-karari.json","ogrenci-karari.json")
    z.write(work/"calisma-kaydi.json","calisma-kaydi.json")
    for module in ['uygulamalar/b11-korelasyon',*modules]:
        for f in sorted((root/module/"sonuclar").rglob("*")):
            if f.is_file() and f.suffix in [".json",".csv"]:z.write(f,str(f.relative_to(root)))
    for figure in new_figures:
        for f in [figure,figure.with_suffix(".png")]: z.write(f,"grafikler/"+f.name)
    for f in (work/"logs").glob("*.log"):z.write(f,"logs/"+f.name)
print("İndirmeye hazır:",archive.name)